In [ ]:
from typing import Any


import pandas as pd

# Load full dataset
data_dir = "/Users/muberraozmen/Development/psycho-pass/experiments/dataset_feb3_1770230537/embeddings_feb18_1771458144/embeddings.parquet"
df = pd.read_parquet(data_dir)

# Filter out to conversations with at least 6 turns
df = df[df["turn_count"] >= 6]

# Filter out to objectives with at least one success and one failure
obj_cat = df["objective"].astype("category")
df["objective_category"] = obj_cat.cat.codes

mixed_objectives = df.groupby("objective_category")[["is_success", "is_failure"]].sum()
mask = (mixed_objectives["is_success"] > 0) & (mixed_objectives["is_failure"] > 0)
mixed_objectives = mixed_objectives[mask].index.tolist()
df = df[df["objective_category"].isin(mixed_objectives)]

# Flatten the dataframe
df["sequence"] = df["turn_count"].apply(lambda x: list(range(x)))
df = df.explode(["sequence", "embeddings"])

# Clean up the dataframe
idx2obj = {i: obj for i, obj in enumerate[Any](obj_cat.cat.categories)}
df = df[["conversation_id", "objective_category", "turn_count", "is_success", "sequence", "embeddings"]]
df["is_last_turn"] = df["sequence"] == df["turn_count"] - 1
df.reset_index(drop=True, inplace=True)


In [ ]:
import numpy as np
from sklearn.manifold import TSNE

# Calculate TSNE
embeddings = np.array(df["embeddings"].to_list())
tsne = TSNE(n_components=2, random_state=42).fit_transform(embeddings)
df['tsne_x'], df['tsne_y'] = tsne[:, 0], tsne[:, 1]

df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create subplots (removed the redundant plt.figure() call)
fig, axs = plt.subplots(1, 2, figsize=(20, 10))

# Define the configurations for each subplot to avoid repetitive code
plot_configs = [
    {"success_flag": True,  "color": "red",  "title": "Successful Attacks", "ax_idx": 0},
    {"success_flag": False, "color": "blue", "title": "Failed Attacks",     "ax_idx": 1}
]

for config in plot_configs:
    ax = axs[config["ax_idx"]]
    
    # Filter base data for this specific plot
    mask = df["is_success"] == config["success_flag"]
    
    # Line plot (draw this first / underneath)
    sns.lineplot(
        data=df[mask],
        x='tsne_x',
        y='tsne_y',
        hue='conversation_id',
        estimator=None,
        palette=[config["color"]],
        marker='o',
        legend=False,
        ax=ax,
        zorder=1  
    )

    # Scatter plot for the last turn (draw this on top)
    sns.scatterplot(
        data=df[mask & df["is_last_turn"]],
        x='tsne_x',
        y='tsne_y',
        marker='X',
        color='black', 
        s=100,
        legend=False,
        ax=ax,
        zorder=2 
    )
    
    # Set subplot formatting
    ax.set_title(config["title"])
    ax.set_xlim(-40, 40)
    ax.set_ylim(-40, 40)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

obj_idx = sorted(df["objective_category"].unique().tolist())
num_plots = len(obj_idx)
fig, axs = plt.subplots(num_plots, 1, figsize=(8, 4* num_plots))

for i, obj in enumerate(obj_idx):  
    df_obj = df[df["objective_category"] == obj].sort_values(by='sequence')
    sns.lineplot(
        data=df_obj,
        x='tsne_x',
        y='tsne_y',
        hue='is_success',
        units='conversation_id',
        estimator=None,
        marker='o',
        palette={False: 'blue', True: 'red'},
        legend=(i == 0),
        ax=axs[i]
    )

    axs[i].set_title(f"Objective {idx2obj[obj]}")

plt.tight_layout()
plt.show()